# BERTje Sentence Analysis

## 1. Preparations

### 1.1 Read Data

In [1]:
import pandas as pd
# read the sentence data 
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx")
# check the head
df.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,1,I was rushed into the [PRESSION] two weeks ago...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",2,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",4,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",5,"I got very nauseous, vomiting, dizzy and rejuv..."


In [2]:
# clean sentences as inputs
sentences = df["Clean_Sentence"].to_list()

### 1.3. Import the list of stopwords

In [3]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/persistent/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg", "coach", "mijnibdcoach", "dr", "uur", "dgs"
]

dutch_stopwords.extend(extra_list)

### 1.4. Embed the lists of sentences with sentence-transformer multilingual-v1

In [4]:
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

/workspace/persistent/mijnidbcoachnlp/tf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-02-13 05:06:44.829212: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739419604.844802  961308 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739419604.849825  961308 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-13 05:06:44.870804: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performa

In [5]:
import numpy as np
# load saved bertje sentence-embeddings # we have two different embeddings due to different calculations 
embeddings = np.load('/workspace/persistent/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl_new_input.npy')


In [6]:
# pre-calculate the sentence embeddings with BERTje if there isn't anything saved 
# calculation 1
#import numpy as np

#embeddings_nl = embedding_model(sentences, truncation=True, padding=True)
#sentence_embeddings_nl = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings_nl])
# save the sentence embeddings

In [7]:
#np.save('/workspace/persistent/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl_new_input.npy', sentence_embeddings_nl)

### 1.5 Pre-configure submodels of BERTopic

In [8]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)

def get_top_words(topic_model):
    topics = topic_model.get_topics()
    top_words = [[word for word, _ in topic[:10]] for topic in topics.values() if topic]  # Top 10 words per topic
    return top_words

def calculate_c_v(topic_model):
    top_words = get_top_words(topic_model)
    coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    return coherence_score

In [9]:
# disable parallelism to avoid some warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 4. Fine-tune the model

In [15]:
import random
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Precompute embeddings (this part depends on your chosen embedding model)
# Example: Assuming 'embeddings' is already computed and available
# embeddings = embedding_model.encode(sentences)

# Initialize BERTopic with the pre-fit vectorizer
topic_model = BERTopic()

# Define parameter ranges
range_n_neighbors = [10, 20, 30, 40, 50]
range_n_components = [2, 3, 4, 5, 6]
range_min_dist = [0.0, 0.05, 0.1]

# Number of random configurations to generate
num_configs = 5  # Adjust this as needed

# Generate random UMAP configurations
umap_configs = [
    {
        "n_neighbors": random.choice(range_n_neighbors),
        "n_components": random.choice(range_n_components),
        "min_dist": random.choice(range_min_dist),
    }
    for _ in range(num_configs)
]

# Define the ranges for HDBSCAN configurations
range_min_cluster_size = [10, 20, 30]  # Example values
range_min_samples = [10, 20, 30]  # Example values

# Generate random HDBSCAN configurations
hdbscan_configs = [
    {
        "min_cluster_size": random.choice(range_min_cluster_size),
        "min_samples": random.choice([x for x in range_min_samples if x <= random.choice(range_min_cluster_size)]),
    }
    for _ in range(num_configs)
]

# Initialize a dictionary to keep track of configurations and their associated coherence scores
tried_configs = {}

# Optionally, load previously tried configurations from a file (if running multiple times)
try:
    with open("tried_configs_bertje.pkl", "rb") as f:
        tried_configs = pickle.load(f)
except FileNotFoundError:
    pass  # If no previous file exists, we start fresh

# Variables to track the best model
best_topic_model = None
best_coherence_score = -float('inf')  # Initializing to a very low score
best_configure = None

# Set the outlier thresholds (upper and lower)
upper_outlier_threshold = 0.5
  # Maximum allowed outlier percentage (5% of points)
lower_outlier_threshold = 0.2  # Minimum allowed outlier percentage (2% of points)

In [16]:
# Fine-tuning loop
for umap_params in umap_configs:
    for hdbscan_params in hdbscan_configs:
        config_tuple = (frozenset(umap_params.items()), frozenset(hdbscan_params.items()))

        # Skip if the configuration has already been tried
        if config_tuple in tried_configs:
            print(f"Skipping already tried configuration: {umap_params}, {hdbscan_params}")
            continue

        # Initialize UMAP and HDBSCAN with current configurations
        umap_model = UMAP(**umap_params)
        hdbscan_model = HDBSCAN(**hdbscan_params)
        print(f"Fitting model for UMAP {umap_params} and HDBSCAN {hdbscan_params}")

        # Update BERTopic with the new UMAP and HDBSCAN models
        topic_model.umap_model = umap_model
        topic_model.hdbscan_model = hdbscan_model
        topic_model.vectorizer_model = CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b')
        topic_model.calculate_probabilities = False

        # Fit the model
        topics, probs = topic_model.fit_transform(sentences, embeddings)

        # Calculate coherence score
        current_coherence_score = calculate_c_v(topic_model)

        # Check if the model has at least 20 topics
        num_topics = len(topic_model.get_topics())
        if num_topics < 20:
            print(f"Skipping model with {num_topics} topics (less than 20).")
            tried_configs[config_tuple] = "skipped"
            continue

        # Check outlier percentage
        outlier_cluster_size = sum(1 for label in hdbscan_model.labels_ if label == -1)
        outlier_percentage = outlier_cluster_size / len(hdbscan_model.labels_)

        if outlier_percentage > upper_outlier_threshold:
            print(f"Skipping model with {outlier_percentage * 100}% outliers (above {upper_outlier_threshold * 100}%).")
            tried_configs[config_tuple] = "skipped"
            continue
        elif outlier_percentage < lower_outlier_threshold:
            print(f"Skipping model with {outlier_percentage * 100}% outliers (below {lower_outlier_threshold * 100}%).")
            tried_configs[config_tuple] = "skipped"
            continue

        # Store the valid configuration and score
        tried_configs[config_tuple] = current_coherence_score

        # Update best model
        if current_coherence_score > best_coherence_score:
            best_coherence_score = current_coherence_score
            best_topic_model = topic_model
            best_configure = (umap_params, hdbscan_params)

        print(f"Coherence score: {current_coherence_score}\n")

        # Save tried configurations after every iteration
        with open("tried_configs_bertje.pkl", "wb") as f:
            pickle.dump(tried_configs, f)


Fitting model for UMAP {'n_neighbors': 10, 'n_components': 3, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 10}
Skipping model with 58.65882410130521% outliers (above 50.0%).
Fitting model for UMAP {'n_neighbors': 10, 'n_components': 3, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 20, 'min_samples': 10}
Skipping model with 59.273752521692636% outliers (above 50.0%).
Skipping already tried configuration: {'n_neighbors': 10, 'n_components': 3, 'min_dist': 0.1}, {'min_cluster_size': 10, 'min_samples': 10}
Skipping already tried configuration: {'n_neighbors': 10, 'n_components': 3, 'min_dist': 0.1}, {'min_cluster_size': 20, 'min_samples': 10}
Fitting model for UMAP {'n_neighbors': 10, 'n_components': 3, 'min_dist': 0.1} and HDBSCAN {'min_cluster_size': 30, 'min_samples': 30}
Skipping model with 19 topics (less than 20).
Fitting model for UMAP {'n_neighbors': 10, 'n_components': 2, 'min_dist': 0.0} and HDBSCAN {'min_cluster_size': 10, 'min_samples': 10}
Coherence

In [1]:
import pickle

# Load the file
with open("tried_configs_bertje.pkl", "rb") as f:
    tried_configs = pickle.load(f)

# Print the contents
for config, score in tried_configs.items():
    print(f"Config: {config}, Score: {score}")


Config: (frozenset({('n_components', 3), ('min_dist', 0.1), ('n_neighbors', 10)}), frozenset({('min_cluster_size', 10), ('min_samples', 10)})), Score: skipped
Config: (frozenset({('n_components', 3), ('min_dist', 0.1), ('n_neighbors', 10)}), frozenset({('min_cluster_size', 20), ('min_samples', 10)})), Score: skipped
Config: (frozenset({('n_components', 3), ('min_dist', 0.1), ('n_neighbors', 10)}), frozenset({('min_cluster_size', 30), ('min_samples', 30)})), Score: skipped
Config: (frozenset({('n_components', 2), ('min_dist', 0.0), ('n_neighbors', 10)}), frozenset({('min_cluster_size', 10), ('min_samples', 10)})), Score: 0.36029988708791105
Config: (frozenset({('n_components', 2), ('min_dist', 0.0), ('n_neighbors', 10)}), frozenset({('min_cluster_size', 20), ('min_samples', 10)})), Score: 0.3631858305275381


In [3]:
from bertopic import BERTopic
topic_model = BERTopic()
# Update BERTopic with the new UMAP and HDBSCAN models
topic_model.umap_model = UMAP(n_neighbors=10, n_components=2, min_dist=0.0)
topic_model.hdbscan_model = HDBSCAN(min_cluster_size=20, min_samples=10)
topic_model.vectorizer_model = CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b')
topic_model.calculate_probabilities = False

# Fit the model
topics, probs = topic_model.fit_transform(sentences, embeddings)

NameError: name 'UMAP' is not defined

In [18]:
best_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,27238,-1_afspraak_gaan_last_goed,"[afspraak, gaan, last, goed, weet, klachten, p...",[Of nog extra 15 tabletten of afwachten tot 20...
1,0,913,0_bekend_uitslag_uitslagen_bloeduitslagen,"[bekend, uitslag, uitslagen, bloeduitslagen, c...",[is de uitslag van de ontlasting inmiddels bek...
2,1,552,1_verstandig_doorgaan_inleveren_prikken,"[verstandig, doorgaan, inleveren, prikken, sto...",[Is het verstandig om het infuus door te laten...
3,2,483,2_slijm_ontlasting_bloed_zit,"[slijm, ontlasting, bloed, zit, soms, dun, hel...",[Sinds 14 dagen heb ik bloed en slijm bij mijn...
4,3,408,3_ivm_mbt_nav_eea,"[ivm, mbt, nav, eea, mvg, tbv, tav, svp, zsm, ...","[Ik had een bericht gestuurd ivm o.a., Ook hb ..."
...,...,...,...,...,...
172,171,20,171_moeilijker_gevoel_drukkend_eet,"[moeilijker, gevoel, drukkend, eet, misselijk,...","[Ben wel gauw vermoeid, weeig gevoel in de maa..."
173,172,20,172_thuis_thuistest_thuistesten_testen,"[thuis, thuistest, thuistesten, testen, voorha...","[Dus nu heb ik geen testen meer thuis., Ik heb..."
174,173,20,173_hoop_betekenen_begrijpt_helpen,"[hoop, betekenen, begrijpt, helpen, antwoord, ...",[Hoop dat jullie iets voor me kunnen betekenen...
175,174,20,174_vertellen_hiervan_uitslagen_aangeven,"[vertellen, hiervan, uitslagen, aangeven, zegg...",[Kunt u mij vertellen wat de uitslag hiervan w...


In [15]:
# check if reduce outliers now work
# Reduce outliers
topics = best_topic_model.topics_
new_topics = best_topic_model.reduce_outliers(sentences, topics)

In [36]:
calculate_c_v(best_topic_model)

0.36392759462116087

In [35]:
best_topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,24540,-1_weet_keer_gaan_afspraak,"[weet, keer, gaan, afspraak, goed, vraag, krijgen, klachten, pijn, medicatie]","[Samengevat voel ik me door de nieuwe medicatie vrij goed., Het gaat wel ok, echter had ik de laatste keer na het prikken wel wat meer last., Mijn laatste vraag gaat over mijn ontlasting.]"
1,0,665,0_verstandig_doorgaan_stoppen_inleveren,"[verstandig, doorgaan, stoppen, inleveren, prikken, afwachten, voren, blijven, raadzaam, laten]","[Is het niet verstandig om vooraf bloed te laten prikken?, Is het misschien verstandig weer eens bloed te laten prikken?, Is het verstandig om eerder te laten bloedprikken/ontlasting inleveren.]"
2,1,525,1_slijm_ontlasting_bloed_dun,"[slijm, ontlasting, bloed, dun, zit, soms, helder, waterig, erbij, spoortje]","[Ik heb geen bloed of slijm bij de ontlasting., Ik heb slijm en bloed bij en na de ontlasting., Ik heb weer slijm maar ook bloed bij de ontlasting.]"
3,2,441,2_ivm_nav_mbt_eea,"[ivm, nav, mbt, eea, mvg, tbv, tav, uitschrijven, svp, ipv]","[Ik heb gelukkig geen verdere klachten tav ibd., We hebben geboekt om mijn ouders samen te kunnen treffen op fuerteventura, dit ivm 20 maanden dat wij elkaar niet gezien hebben ivm Corona., Ik had..."
4,3,329,3_ziekenhuis_zorginstelling_opgenomen_ophalen,"[ziekenhuis, zorginstelling, opgenomen, ophalen, apotheek, prikken, herstelzorg, terugvinden, gestuurd, dienst]","[Het was in het [ZIEKENHUIS]., Gewoon in het [ZIEKENHUIS]., Dit was rond 13 maart in het [ZIEKENHUIS] te .]"
...,...,...,...,...,...
270,269,15,269_vergoeden_apotheek_recept_liet,"[vergoeden, apotheek, recept, liet, genoteerd, belde, aangekomen, ontvangen, heden, gaf]","[Recept is nog niet bij de apotheek?, Nog steeds geen recept binnen bij apotheek!, De apotheek liet weten deze niet te vergoeden.]"
271,270,15,270_calcium_kauwtabletten_herhaalrecept_mesalazine,"[calcium, kauwtabletten, herhaalrecept, mesalazine, kauwtablet, thiosix, mesavant, reguliere, methamucil, calciumvitamine]","[Ik had graag een herhaalrecept voor de thiosix en de kauwtabletten calcium., ik had graag een herhaalrecept voor de thiosix en de kauwtabletten calcium., Ik had graag een herhaalrecept voor de th..."
272,271,15,271_advies_actuele_feedback_graag,"[advies, actuele, feedback, graag, mening, hulp, hierover, informatie, , ]","[Graag advies van jullie., Graag jullie advies., Graag jullie advies.]"
273,272,15,272_overdag_last_gelukkig_extreem,"[overdag, last, gelukkig, extreem, direct, slecht, stress, daarvoor, hiervan, geval]","[Overdag heb ik in ieder geval geen last van koorts., Overdag heb ik nergens last van., Overdag heb ik gelukkig nog geen last.]"
